# AI Engineering Interview Prep
## Section: LangGraph

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


In [ ]:
# ── Optional: Install dependencies for runnable code cells ────────────────
# Uncomment and run this cell first if you want to execute the code examples

# !pip install sentence-transformers faiss-cpu numpy pydantic langchain
# !pip install langchain-community langchain-google-genai
# !pip install groq openai anthropic
# !pip install ragas datasets evaluate

print("📚 AI Engineering Interview Prep Notebook")
print("✅ Ready to study! All 491+ questions are answered below.")
print()
print("Sections:")
sections = [
    "1. LLM Fundamentals (63 Qs)",
    "2. Prompt Engineering (30 Qs)",
    "3. RAG (37 Qs)",
    "4. AI Agents (37 Qs)",
    "5. Fine-Tuning (25 Qs)",
    "6. Vector Databases (22 Qs)",
    "7. AI System Design (42 Qs)",
    "8. LLMOps (41 Qs)",
    "9. Evaluation (29 Qs)",
    "10. AI Safety & Ethics (38 Qs)",
    "11. Multimodal AI (26 Qs)",
    "12. AI Infrastructure (27 Qs)",
    "13. Coding Practicals (22 Qs)",
    "14. Behavioral Questions (22 Qs)",
    "15. LangChain (10 Qs)",
    "16. LangGraph (10 Qs)",
    "17. Resume-Based Questions (10 Qs)",
    "⚡ Quick Reference Cheat Sheet",
]
for s in sections:
    print(f"  📌 Section {s}")


---
## 🕸️ Section 16 — LangGraph

### Q1. What is LangGraph, and when would you use it over standard LangChain agents?
**A:** Standard LangChain agents (like the legacy `AgentExecutor`) use a linear, hard-coded loop (LLM decides tool → runs tool → LLM decides next step). They struggle with complex, cyclical workflows where you need state, conditional loops, or multi-agent collaboration. LangGraph is a framework that lets you build agentic systems as a graph of nodes (computations/LLM calls) and edges (control flow). You use LangGraph when your agent needs explicit state management, human approval steps, parallel execution branches, or cyclical loops (like an agent writing code, compiling it, catching errors, and revising in a loop).

### Q2. How does state management and "reducers" work in LangGraph?
**A:** Every graph in LangGraph has a shared schema called the `State` (usually a TypedDict). Each node receives the current state as input and returns a dictionary updating specific keys. Crucially, how these updates are applied is controlled by "reducers." By default, returning a key overwrites its value. But for keys like a list of chat messages, you use a reducer like `Annotated[list, add_messages]`. This tells LangGraph to *append* or *merge* updates rather than overwrite them, preventing you from accidentally wiping out the conversation history.

### Q3. Explain Nodes, Edges, and Conditional Edges in LangGraph.
**A:** 
- **Nodes:** Plain Python functions (sync or async) that take the current `State` as an argument, perform some logic (like an LLM call or tool execution), and return a state update.
- **Edges:** Control flow lines. A standard edge (`workflow.add_edge("node_A", "node_B")`) means node B always executes immediately after node A.
- **Conditional Edges:** Routing functions that check the state and return the next node's name. For example, after an LLM call, a conditional edge checks if the output contains a tool call; if yes, it routes to the tool node; if no, it routes to the end node.

### Q4. How does LangGraph support memory persistence and checkpointing?
**A:** LangGraph has built-in persistence using checkpointers (like `MemorySaver` for in-memory or `SqliteSaver`/`PostgresSaver` for databases). When you compile your graph with a checkpointer, LangGraph automatically saves the state at every single step of the graph's execution. To use it, you pass a `thread_id` in the config. When a new message comes in under the same `thread_id`, the checkpointer restores the exact state at the last step. This enables robust multi-turn conversations and allows graph executions to resume seamlessly if the server crashes mid-run.

### Q5. How do you implement Human-in-the-Loop (Interrupts) in LangGraph?
**A:** In LangGraph, you can pause graph execution before entering a specific node by adding a `breakpoint` in your compile configuration. When compiled with a checkpointer, the graph runs until it hits the breakpoint, saves the state, and pauses. The frontend or admin can review the proposed tool call or action. Once approved, you resume execution by invoking the graph with the same `thread_id` and passing a `None` or updated input. It allows for safe human validation before high-risk actions (like sending an email or deleting data).

### Q6. What is "Time Travel" and "State Replay" in LangGraph, and how does it help debugging?
**A:** Because LangGraph's checkpointer saves the state at *every* step, you can retrieve the full history of state updates for any thread. "Time Travel" is the ability to query past checkpoints, see what the state looked like at step 3, and even fork the execution from that past step. If your agent hallucinated at step 5 of a complex run, you don't need to rerun the whole sequence. You can inspect step 4, modify the state (e.g., edit the prompt or tool output), and resume execution from there. It's a game-changer for debugging production agent failures.

### Q7. How do you handle parallel node execution and fan-out/fan-in in LangGraph?
**A:** You can define a node to route to multiple nodes simultaneously by returning a list of node names in a conditional edge. LangGraph will run these nodes in parallel (useful for generating multiple search queries or running multiple sub-agents). Once all parallel nodes finish, they write their updates to the state. The reducers automatically handle merging these updates. The graph then "fans-in" to a downstream joining node. This parallelization is handled natively under the hood without needing custom threading code.

### Q8. How does streaming work in LangGraph, and what is the difference between "values" and "updates" streaming?
**A:** LangGraph supports streaming of both intermediate state steps and LLM tokens. When calling `.stream()`, you specify the streaming mode:
- `values` mode: streams the entire state dictionary after every step. Great for updating the frontend with the complete updated conversation history.
- `updates` mode: streams *only* the specific dictionary updates returned by the node that just executed. Useful if you want to log or display which specific action was just taken (e.g., "Tool execution completed").

### Q9. How do you implement modularity using Subgraphs in LangGraph?
**A:** For large agentic systems, a single massive graph becomes unreadable and hard to maintain. LangGraph lets you compile a graph and then use it as a node inside a parent graph. The parent state and subgraph state can be mapped so they share keys. For example, you can have a parent router graph that delegates to a specialized "coding subgraph" or "RAG subgraph." Each subgraph manages its own internal loops and state, keeping the parent architecture clean and modular.

### Q10. How do you prevent infinite loops in cyclical LangGraph agents?
**A:** When an agent gets stuck in a loop (e.g., tool fails -> agent retries with same inputs -> tool fails again), it can drain API credits rapidly. LangGraph handles this by accepting a `recursion_limit` parameter in the execution `config` (default is often 25). If the graph executes more than 25 steps without reaching the `__end__` node, LangGraph terminates execution and raises a `GraphRecursionError`. In production, you catch this error to return a graceful fallback message to the user, ensuring your agent never loops infinitely.